# RQ4 Generalization

How do effectiveness and efficiency vary across VLM families and tasks? The notebook reproduces the per-SUT and per-task strata, the transfer matrix of crossings between SUTs, the agreement with greedy generation and the further analysis of cell-level predictors of hardness.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display
from mmb.analysis import stats as st, paired as pr, export as ex

RESULTS = Path("../results") if Path("../results").exists() else Path("results")
FIG = RESULTS / "figures"; FIG.mkdir(parents=True, exist_ok=True)
GRID = ["qwen", "internvl", "ministral"]      # the three SUTs of the study (balanced grid)
REF = "a0_directed"                            # MMBA, the reference of every paired test
N_BOOT = 2000                                  # the paper uses 10000; raise for the final numbers
df = st.load_results([RESULTS / "MAIN.csv"])
scopes = ex.scopes(df, GRID)                   # {(sut, task): frame}; ("All", "All") pools the grid
grid_df = scopes[("All", "All")]
print(len(df), "runs;", "arms:", sorted(df["arm"].unique()))

## Per SUT and task

The summary of RQ1 for every (SUT, task) stratum.

In [ ]:
rows = []
for (sut, task), d in scopes.items():
    s = st.arm_summary(d, N_BOOT); s.insert(0, "task", task); s.insert(0, "sut", sut); rows.append(s)
(pd.concat(rows, ignore_index=True)[["sut", "task", "arm", "n_runs", "share_crossed", "wilson_lo", "wilson_hi"]] if rows else None)

## Directed against random on each SUT and on both tasks

The two arms on the four SUTs (ImageNet) and on the two tasks (grid SUTs pooled).

In [ ]:
prim = df[df["arm"].isin([REF] + [a for a, _ in ex.PRIMARY_ARMS])]
for sut in sorted(set(prim["sut"])):
    d = prim[(prim["sut"] == sut) & (prim["task"] == "imagenet")]
    if len(d): print(sut); display(st.arm_summary(d, N_BOOT))

## Transfer of crossings between SUTs

Every crossing input replayed on the other SUTs against that SUT's own anchor. The files come from the cross-SUT replay script; the cell is skipped until they exist.

In [ ]:
from mmb.analysis import transfer as tr
files = sorted(RESULTS.glob("transfer_MAIN*.csv"))
if files:
    tdf = tr.load_transfer(files, results=RESULTS / "MAIN.csv")
    display(pd.DataFrame(tr.matrix_rows(tdf)))
else:
    print("no transfer replay files yet")

## Agreement with greedy generation

Share of crossing runs whose greedy answer equals the teacher-forced argmax, and whether the single-item re-scoring keeps the sign of the crossing.

In [ ]:
files = sorted(RESULTS.glob("agree_MAIN_*.csv"))
if files:
    adf = ex.load_agree(files)
    display(pd.DataFrame(ex.agree_rows(adf)))
else:
    print("no agreement replay files yet")

## Further analysis: what makes a boundary hard to reach

Spearman correlation and AUC of cell-level quantities (anchor margin, answer-string prior, generation-zero probe) with the floor and the crossing. The relations are rank orders, not predictions.

In [ ]:
from mmb.analysis import predictors as pred, plots
a0 = df[df["arm"] == REF]
if len(a0) and a0["prior_g"].notna().any():
    cells = pred.cell_frame(pred.prepare(a0), ["sut", "task"])
    tab, corr, part = ex.e2b_table(cells, N_BOOT)
    display(corr)
    plots.plot_predictor_scatter(df[df["sut"].isin(GRID)], FIG / "predictor-main.pdf", arm=REF)
else:
    print("no directed-arm runs with prior and probe columns yet")